# FOG on Kaggle

Use a GPU notebook with Internet enabled for installation. Attach the generated FOG bundle as a dataset. This notebook keeps Kaggle's CUDA PyTorch and uses a separate environment for FOG. Missing training data must be restored or regenerated before running the job.


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import zipfile

WORK = Path("/kaggle/working/fog")
INPUT = Path("/kaggle/input")
if not (WORK / "bundle_manifest.json").is_file():
    manifests = list(INPUT.rglob("bundle_manifest.json"))
    archives = list(INPUT.rglob("fog-kaggle-input.zip"))
    if len(manifests) == 1:
        if WORK.exists():
            raise RuntimeError("The work directory already exists without a bundle manifest; inspect it first.")
        shutil.copytree(manifests[0].parent, WORK)
    elif not manifests and len(archives) == 1:
        with zipfile.ZipFile(archives[0]) as archive:
            for name in archive.namelist():
                target = (WORK.parent / name).resolve()
                if not target.is_relative_to(WORK.resolve()):
                    raise ValueError(f"Unexpected archive path: {name}")
            archive.extractall(WORK.parent)
    else:
        raise RuntimeError("Attach exactly one FOG bundle dataset, then rerun this cell.")
os.chdir(WORK)
print(json.loads((WORK / "bundle_manifest.json").read_text()))


In [ ]:
# Installation needs Internet. The dataset already contains the built simulator.
if not shutil.which("node") or not shutil.which("npm"):
    subprocess.run(["apt-get", "update"], check=True)
    subprocess.run(["apt-get", "install", "-y", "nodejs", "npm"], check=True)
subprocess.run([sys.executable, "-m", "fog.scripts.kaggle_setup"], cwd=WORK, check=True)
PYTHON = str(WORK / ".venv/bin/python")


In [ ]:
# Restore missing inputs at their repository-relative paths before continuing.
# This rechecks the actual files, not just the inventory captured when the bundle was made.
check = """
from pathlib import Path
from fog.scripts.kaggle_bundle import bundle_files
bundle_files(Path("configs/automated_gen9ou_typed_league.yaml").resolve(), Path.cwd())
print("Training inputs are present.")
"""
subprocess.run([PYTHON, "-c", check], check=True)


In [ ]:
# Keep hardware-specific settings in a new job; retain the original recipe.
JOB = WORK / "configs/kaggle_job.yaml"
prepare = """
from pathlib import Path
import os, yaml
source = Path("configs/automated_gen9ou_typed_league.yaml")
job = yaml.safe_load(source.read_text())
job["name"] = "gen9ou_typed_kaggle"
job["output_dir"] = "runs/gen9ou_typed_kaggle"
job["training"]["minibatch_size"] = 512
job.setdefault("overrides", {}).update({
    "device": "cuda",
    "env.rollout_workers": min(2, os.cpu_count() or 1),
    "env.actor_threads": 1,
    "env.node_binary": "node",
    "env.showdown_root": "server/pokemon-showdown",
    "preflight.report": "runs/preflight_audit/kaggle/summary.json",
    "preflight.replay_report": "runs/replay_preflight_audit/kaggle/summary.json",
})
destination = Path("configs/kaggle_job.yaml")
text = yaml.safe_dump(job, sort_keys=False)
if destination.exists() and destination.read_text() != text:
    raise RuntimeError("Existing Kaggle job differs; inspect it before changing a resumed run.")
destination.write_text(text)
"""
subprocess.run([PYTHON, "-c", prepare], check=True)
subprocess.run([PYTHON, "-m", "fog.scripts.automated_training", "run",
                "--job", str(JOB), "--dry-run"], check=True)


The first training block refreshes both zero-update audits before PPO. Leave their gates enabled. `--max-blocks 1` bounds this session to one block; rerunning continues from saved job state. Recovery after an interrupted block is a weight-based continuation, not guaranteed bit-identical optimizer continuation.


In [ ]:
RUN_TRAINING = False  # Change to True when the input check passes and setup is complete.
MAX_BLOCKS = 1
if RUN_TRAINING:
    environment = os.environ.copy()
    environment.update(OMP_NUM_THREADS="1", MKL_NUM_THREADS="1")
    subprocess.run([PYTHON, "-m", "fog.scripts.automated_training", "run",
                    "--job", str(JOB), "--max-blocks", str(MAX_BLOCKS)],
                   env=environment, check=True)
else:
    print("Setup complete. Training has not started.")


In [ ]:
# Run this after training or an interruption, then save the notebook version with outputs.
OUTPUT = WORK / "runs/gen9ou_typed_kaggle"
if OUTPUT.exists():
    shutil.copy2(JOB, OUTPUT / "job.yaml")
    shutil.copytree(WORK / "runs/kaggle_environment", OUTPUT / "environment", dirs_exist_ok=True)
    for audit in ("preflight_audit", "replay_preflight_audit"):
        report = WORK / "runs" / audit / "kaggle/summary.json"
        if report.exists():
            shutil.copy2(report, OUTPUT / f"{audit}.json")
    subprocess.run([PYTHON, "-m", "fog.scripts.automated_training", "export",
                    "--job", str(JOB), "--out", "/kaggle/working/fog-training-output.zip"], check=True)
else:
    print("No training output yet.")
